In [1]:
import pandas as pd

In [2]:
# Cleaned data
energy_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\energy_df.xlsx')
material_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\material_df.xlsx')
biosphere_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\biosphere_df.xlsx')

In [3]:
# Prices and production data
price_df = pd.read_excel(r'data/Prices/Prices_data.xlsx', sheet_name='data')
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [4]:
from utils.data_manipulations import build_activity_name, add_site_id

In [5]:
# Keep only relevant columns
energy_df = energy_df[['main_id', 'facility_group_id', 'flow_type', 'subflow_type', 'value_MJ']]
material_df = material_df[['main_id', 'facility_group_id', 'flow_type', 'subflow_type', 'mass_t']]
biosphere_df = biosphere_df[['main_id', 'facility_group_id', 'compartment_name', 'substance_name', 'flow_direction', 'release_pathway', 'unit', 'value']]

In [6]:
# Add activitiy_name to production_df
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)

In [7]:
production_df = add_site_id(production_df)
energy_df = add_site_id(energy_df)
material_df = add_site_id(material_df)
biosphere_df = add_site_id(biosphere_df)

In [8]:
energy_df = energy_df.merge(production_df[['site_id', 'activity_name']], on='site_id', how='left')
material_df = material_df.merge(production_df[['site_id', 'activity_name']], on='site_id', how='left')
biosphere_df = biosphere_df.merge(production_df[['site_id', 'activity_name']], on='site_id', how='left')

In [9]:
# Replace column name mass_t to mass for normalization function
material_df = material_df.rename(columns={'mass_t': 'mass'})

# Data-gap filling

In [21]:
production_df

,main_id,facility_group_id,facility_name,facility_group_name,province,facility_type,mining_processing_type,archetypes,biosphere_data?,technosphere_data?,...,Mo_conc,Zn_conc,Pb_conc,Fe_conc,Pt_conc,Pd_conc,U_conc,Nb_conc,activity_name,site_id
0,QC-MAIN-089f3c60,<NA>,Bloom Lake,NaN,Quebec,mining,Open-pit,Magnetite concentrator,NPRI+GHG+Land,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,None,QC-MAIN-089f3c60
1,BC-MAIN-857b7b89,<NA>,Brucejack,NaN,British Columbia,mining,"Underground, concentrator",NaN,NPRI+GHG+Water,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Underground mining and beneficiation at Br...",BC-MAIN-857b7b89
2,QC-MAIN-de3d8b7b,<NA>,Canadian Electrolytic Zinc Limited (CEZinc),NaN,Quebec,manufacturing,Refinery,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Zn and Pb, refining at Canadian Electrolytic Z...",QC-MAIN-de3d8b7b
3,QC-MAIN-e7e6a960,<NA>,Canadian Malartic,NaN,Quebec,mining,"Open-pit, concentrator",Hybrid free-miling-refactory,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au and Ag, Open-pit mining and beneficiation a...",QC-MAIN-e7e6a960
4,NL-MAIN-dd723db4,<NA>,Carol Lake,NaN,Newfoundland and Labrador,mining,"Open-pit, concentrator",Fe concentrator + pellet plant,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,None,NL-MAIN-dd723db4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,<NA>,GRP-0a2c0d69,NaN,Meadowbank complex,Nunavut,mining,"Open-pit, underground",Free-miling,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au and Ag, Open-pit and underground mining at ...",GRP-0a2c0d69
62,<NA>,GRP-0d911886,NaN,Porcupine complex,Ontario,mining,"Open-pit, underground",Free-miling,NPRI+GHG+Land,Energy and materials,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Open-pit and underground mining at Porcupi...",GRP-0d911886
63,<NA>,GRP-147b3123,NaN,Timmins Operation,Ontario,mining,"Underground, concentrator",Free-miling,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Underground mining and beneficiation at Ti...",GRP-147b3123
64,<NA>,GRP-14bfbb82,NaN,Seabee Gold Operation,Saskatchewan,mining,"Underground, concentrator",Free-miling,NPRI+GHG+Land,Energy,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,"Au, Underground mining and beneficiation at Se...",GRP-14bfbb82


## Normalize flows

In [12]:
from core.normalization import normalize_flows

### Per ore processed

In [16]:
energy_ore = normalize_flows(energy_df, production_df, mode='ore', value_col='value_MJ')

In [17]:
material_ore = normalize_flows(material_df, production_df, mode='ore', value_col='mass')

In [18]:
biosphere_ore = normalize_flows(biosphere_df, production_df, mode='ore', value_col='value')

### Per concentrate stream

In [19]:
energy_conc_econ = normalize_flows(energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value_MJ')

In [20]:
energy_conc_econ

,main_id,facility_group_id,flow_type,subflow_type,value_MJ,site_id,activity_name,concentrate,mass_conc,allocation_factor,facility_type,value_normalized,functional_unit,normalization_key
0,BC-MAIN-599152a0,<NA>,Energy,Diesel,8.096000e+07,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Cu,63500.000000,1.000000,mining,1274.960630,Cu concentrate,concentrate_economic
1,BC-MAIN-599152a0,<NA>,Energy,Electricity consumption,1.968000e+09,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Cu,63500.000000,1.000000,mining,30992.125984,Cu concentrate,concentrate_economic
2,BC-MAIN-599152a0,<NA>,Energy,Gasoline,3.680000e+05,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Cu,63500.000000,1.000000,mining,5.795276,Cu concentrate,concentrate_economic
3,BC-MAIN-599152a0,<NA>,Energy,Propane,1.012000e+07,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Cu,63500.000000,1.000000,mining,159.370079,Cu concentrate,concentrate_economic
4,BC-MAIN-6b4800fe,<NA>,Energy,Diesel,1.683653e+09,BC-MAIN-6b4800fe,"Cu and Mo, Open-pit mining and beneficiation a...",Cu,185367.930667,0.999756,mining,9080.543852,Cu concentrate,concentrate_economic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,<NA>,GRP-a13779f8,Energy,Electricity consumption,7.640000e+08,GRP-a13779f8,"Au and Ag and Cu and Zn, Underground mining an...",Zn,69284.000000,0.389976,mining,4300.290728,Zn concentrate,concentrate_economic
61,<NA>,GRP-a13779f8,Energy,Gasoline,1.760000e+06,GRP-a13779f8,"Au and Ag and Cu and Zn, Underground mining an...",Cu,40513.333333,0.610024,mining,26.500979,Cu concentrate,concentrate_economic
62,<NA>,GRP-a13779f8,Energy,Gasoline,1.760000e+06,GRP-a13779f8,"Au and Ag and Cu and Zn, Underground mining an...",Zn,69284.000000,0.389976,mining,9.906429,Zn concentrate,concentrate_economic
63,<NA>,GRP-a13779f8,Energy,Propane,4.840000e+07,GRP-a13779f8,"Au and Ag and Cu and Zn, Underground mining an...",Cu,40513.333333,0.610024,mining,728.776912,Cu concentrate,concentrate_economic


In [ ]:
material_conc_econ = normalize_flows(energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value_MJ')

### Per metal produced

In [28]:
energy_metal_econ = normalize_flows(energy_df, production_df, price_df=price_df, mode='metal', allocation='economic', value_col='value_MJ')

In [29]:
energy_metal_econ

,main_id,facility_group_id,flow_type,subflow_type,value_MJ,site_id,activity_name,metal,mass_t,allocation_factor,facility_type,value_normalized,functional_unit,normalization_key
0,BC-MAIN-599152a0,<NA>,Energy,Diesel,8.096000e+07,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Au,0.275204,0.094859,mining,2.790574e+07,"Au, usable ore",metal_economic
1,BC-MAIN-599152a0,<NA>,Energy,Diesel,8.096000e+07,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Ag,6.789334,0.030369,mining,3.621385e+05,"Ag, usable ore",metal_economic
2,BC-MAIN-599152a0,<NA>,Energy,Diesel,8.096000e+07,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Cu,19050.000000,0.874772,mining,3.717667e+03,"Cu, usable ore",metal_economic
3,BC-MAIN-599152a0,<NA>,Energy,Electricity consumption,1.968000e+09,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Au,0.275204,0.094859,mining,6.783412e+08,"Au, usable ore",metal_economic
4,BC-MAIN-599152a0,<NA>,Energy,Electricity consumption,1.968000e+09,BC-MAIN-599152a0,"Au and Ag and Cu, Open-pit mining and benefici...",Ag,6.789334,0.030369,mining,8.802971e+06,"Ag, usable ore",metal_economic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,<NA>,GRP-147b3123,Energy,Dynamite,4.889800e+06,GRP-147b3123,"Au, Underground mining and beneficiation at Ti...",Au,4.133655,1.000000,mining,1.182924e+06,"Au, usable ore",metal_economic
292,<NA>,GRP-147b3123,Energy,Electricity consumption|Grid electricity,7.382268e+08,GRP-147b3123,"Au, Underground mining and beneficiation at Ti...",Au,4.133655,1.000000,mining,1.785894e+08,"Au, usable ore",metal_economic
293,<NA>,GRP-147b3123,Energy,Emulsions,2.026300e+06,GRP-147b3123,"Au, Underground mining and beneficiation at Ti...",Au,4.133655,1.000000,mining,4.901957e+05,"Au, usable ore",metal_economic
294,<NA>,GRP-147b3123,Energy,Gasoline,2.946100e+06,GRP-147b3123,"Au, Underground mining and beneficiation at Ti...",Au,4.133655,1.000000,mining,7.127106e+05,"Au, usable ore",metal_economic


# Exports normalized dataframes

In [66]:
energy_norm_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore_normalization/energy_df.csv', index=False)
material_norm_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore_normalization/material_df.csv', index=False)
biosphere_norm_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore_normalization/biosphere_df.csv', index=False)

In [67]:
energy_norm_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/economic_allocation/energy_df.csv', index=False)
material_norm_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/economic_allocation/material_df.csv', index=False)
biosphere_norm_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/economic_allocation/biosphere_df.csv', index=False)